# putEMG — Efficient Feature-Based Model (8-Channel LOSO)

Retrains the LOSO cross-subject evaluation from `experiments/cross_subject/feature_based/`
but using only the **8 representative channels** selected by feature-based agglomerative
clustering (see `experiments/clustering/clustering_features.ipynb`).

**Goal:** match or closely approach the 24-channel feature-based accuracy with one-third
the electrode count — demonstrating that the discriminative channel selection works for
hand-crafted features, not just deep learning.

| Input | 24-channel baseline | This notebook |
|-------|-------------------|--------------|
| Shape | `(N, 26, 3192)` | `(N, 26, ≈1064)` |
| Source | `load_feature_subjects` | slice raw → extract 8-ch features |

**Prerequisites:**
1. `data_preprocessing/driver.ipynb` — per-subject `.mat` files
2. `experiments/clustering/clustering_features.ipynb` — `feat_representative_channels.npy`

Set `MODEL_TYPE` in the config cell to switch between `FeatureLSTM`, `FeatureGRU`, `FeatureTransformer`.
Completed folds are skipped automatically — safe to stop and resume.

In [ ]:
import os
import sys
import numpy as np
import datetime
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
# ── Shared utilities (src/) ────────────────────────────────────────────────────
sys.path.append(os.path.abspath('../../../'))

import src.feature_based_models as fbm
from src.emg_loader import load_all_subjects, load_feature_subjects, make_loso_train_val_test
from src.feature_extraction import batch_extract

In [ ]:
# ── Load channel selection from feature clustering notebook ────────────────────
CLUSTERING_DIR = os.path.abspath('../../clustering')
channels_path  = os.path.join(CLUSTERING_DIR, 'feat_representative_channels.npy')

if not os.path.exists(channels_path):
    raise FileNotFoundError(
        f'Channel selection file not found: {channels_path}\n'
        'Run experiments/clustering/clustering_features.ipynb first.'
    )

CHANNELS = np.load(channels_path)   # (8,) — 0-indexed channel indices
N_CH     = len(CHANNELS)
print(f'Using {N_CH} channels (0-indexed): {list(CHANNELS)}')
print(f'(1-indexed): {[c + 1 for c in CHANNELS]}')

In [ ]:
def train(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(train_loader)


def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            correct += (model(X).argmax(dim=1) == y).sum().item()
            total   += y.size(0)
    return correct / total

In [ ]:
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Using device: {device}')

In [ ]:
_notebook_dir = os.path.abspath(os.getcwd())

# ── Config ────────────────────────────────────────────────────────────────────
DATA_DIR        = '/Volumes/KRIS/data/UG_per_subject'
# Separate feature cache for 8-channel subjects — keeps full-channel cache intact
FEATURE_DIR_8CH = '/Volumes/KRIS/data/features_8ch_sequence'
MODE            = 'sequence'

MODEL_TYPE  = 'FeatureLSTM'   # 'FeatureGRU' | 'FeatureTransformer'
WEIGHTS_DIR = os.path.join(_notebook_dir, 'weights', MODEL_TYPE)
RESULTS_DIR = os.path.join(_notebook_dir, 'results')
LOG_PATH    = os.path.join(RESULTS_DIR, f'results_log_{MODEL_TYPE}.txt')

BATCH_SIZE = 16
VAL_FRAC   = 0.10
MAX_EPOCHS = 20
PATIENCE   = 5
MIN_DELTA  = 0.002
LR         = 1e-3
DROPOUT    = 0.3

os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FEATURE_DIR_8CH, exist_ok=True)
print(f'Weights → {WEIGHTS_DIR}')
print(f'Log     → {LOG_PATH}')

---
## Feature Extraction (8 Channels)

Slices raw subjects to the 8 representative channels, then extracts libemg features.
Output is cached in `FEATURE_DIR_8CH` — subjects with existing files are skipped.
The `INPUT_SIZE` for the models is derived dynamically from the loaded data shape.

In [ ]:
# Slice raw subjects to 8 representative channels before extraction
# X per subject: (N_reps, 1, 24, 1500) → (N_reps, 1, 8, 1500)
raw_subjects_full = load_all_subjects(DATA_DIR)
subjects_8ch_raw  = [(name, X[:, :, CHANNELS, :], y) for name, X, y in raw_subjects_full]

expected = len(subjects_8ch_raw)
present  = len([f for f in os.listdir(FEATURE_DIR_8CH) if f.endswith(f'_{MODE}.npz')])
print(f'Subjects          : {expected}')
print(f'Cached (8-ch)     : {present}  ({FEATURE_DIR_8CH})')

if present < expected:
    print(f'\nExtracting 8-channel features for {expected - present} subject(s)...')
    batch_extract(subjects_8ch_raw, output_dir=FEATURE_DIR_8CH, mode=MODE)
else:
    print('All 8-channel feature files present — skipping extraction.')

subjects   = load_feature_subjects(FEATURE_DIR_8CH, mode=MODE)
INPUT_SIZE = subjects[0][1].shape[2]   # (N_reps, 26, input_size) — computed from data
print(f'\nInput size (8-channel features): {INPUT_SIZE}')
print(f'Subject 0 X shape: {subjects[0][1].shape}')

---
## LOSO Training Loop

Identical protocol to `experiments/cross_subject/feature_based/` — only the
feature dimensionality changes from ~3192 (24 channels) to ~1064 (8 channels).

- **Test** — 1 held-out subject, never seen during training
- **Train / Val** — all other 43 subjects, split 90/10 (stratified by class, seeded)
- Checkpointing: fold skipped if `weights/<MODEL_TYPE>/<MODEL_TYPE>_<id>.pt` already exists

In [ ]:
MODEL_MAP = {
    'FeatureLSTM':        fbm.FeatureLSTM,
    'FeatureGRU':         fbm.FeatureGRU,
    'FeatureTransformer': fbm.FeatureTransformer,
}


def subject_id(filename):
    # 'features_subject_03_sequence.npz' -> '03'
    return filename.split('_')[2]


def weight_path(subject_name):
    return os.path.join(WEIGHTS_DIR, f'{MODEL_TYPE}_{subject_id(subject_name)}.pt')


def update_log():
    checkpoints = []
    for fname in sorted(os.listdir(WEIGHTS_DIR)):
        if not fname.endswith('.pt'):
            continue
        ckpt = torch.load(os.path.join(WEIGHTS_DIR, fname), map_location='cpu', weights_only=False)
        if 'subject' not in ckpt:
            continue
        checkpoints.append(ckpt)

    if not checkpoints:
        return

    accs   = [c['test_acc'] * 100 for c in checkpoints]
    n_done = len(checkpoints)

    lines = [
        f'putEMG — {MODEL_TYPE} LOSO Results  (8-channel efficient feature-based model)',
        '=' * 72,
        f"{'Subject':<32} {'Test Acc':>9}  {'Val Acc':>9}  {'Epoch':>6}  {'Date'}",
        '-' * 72,
    ]
    for c in checkpoints:
        lines.append(
            f"{c['subject']:<32} {c['test_acc']*100:>8.2f}%  "
            f"{c['val_acc']*100:>8.2f}%  {c['best_epoch']:>6}  {c['date']}"
        )
    lines += [
        '=' * 72,
        f"Mean: {np.mean(accs):.2f}%  \u00b1  {np.std(accs):.2f}%  "
        f"({n_done} / {len(subjects)} folds complete)",
    ]

    with open(LOG_PATH, 'w') as f:
        f.write('\n'.join(lines) + '\n')

    print(f'Log updated \u2192 {LOG_PATH}  ({n_done}/{len(subjects)} folds)')

In [ ]:
for test_idx, (test_name, _, _) in enumerate(subjects):
    wpath = weight_path(test_name)

    if os.path.exists(wpath):
        print(f'[SKIP] {test_name}  — checkpoint found')
        continue

    print(f"\n{'='*60}")
    print(f'  Fold {test_idx+1}/{len(subjects)}  —  test: {test_name}')
    print(f"{'='*60}")

    train_loader, val_loader, test_loader = make_loso_train_val_test(
        subjects, test_idx, val_frac=VAL_FRAC, batch_size=BATCH_SIZE
    )

    # Pass INPUT_SIZE so the model matches the 8-channel feature dimensionality
    model     = MODEL_MAP[MODEL_TYPE](input_size=INPUT_SIZE, dropout_rate=DROPOUT).to(device)
    optimizer = optim.Adam(model.parameters(), lr=LR)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, min_lr=1e-6)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = float('-inf')
    best_state   = None
    best_epoch   = 0
    bad_epochs   = 0

    for epoch in range(MAX_EPOCHS):
        tr_loss = train(model, train_loader, criterion, optimizer, device)
        val_acc = evaluate(model, val_loader, device)
        curr_lr = optimizer.param_groups[0]['lr']

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state   = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_epoch   = epoch + 1

        scheduler.step(val_acc)
        bad_epochs = 0 if val_acc >= (best_val_acc - MIN_DELTA) else bad_epochs + 1

        print(f'  Epoch {epoch+1:3d}: loss={tr_loss:.4f}  val={val_acc*100:.2f}%  '
              f'best={best_val_acc*100:.2f}%  lr={curr_lr:.2e}')

        if bad_epochs >= PATIENCE:
            print(f'  Early stopping at epoch {epoch+1}.')
            break

    model.load_state_dict(best_state)
    test_acc = evaluate(model, test_loader, device)
    print(f'\n  Test accuracy: {test_acc*100:.2f}%')

    torch.save({
        'subject':    test_name,
        'test_acc':   test_acc,
        'val_acc':    best_val_acc,
        'best_epoch': best_epoch,
        'channels':   list(CHANNELS),
        'n_channels': N_CH,
        'input_size': INPUT_SIZE,
        'dropout':    DROPOUT,
        'state_dict': best_state,
        'date':       datetime.date.today().isoformat(),
    }, wpath)

    update_log()

print(f"\n{'='*60}")
print('  All folds complete.')
print(f"{'='*60}")
update_log()

---
## Results

In [ ]:
if os.path.exists(LOG_PATH):
    with open(LOG_PATH) as f:
        print(f.read())
else:
    print('No results yet — run the training loop first.')

In [ ]:
checkpoints = []
for fname in sorted(os.listdir(WEIGHTS_DIR)):
    if not fname.endswith('.pt'):
        continue
    ckpt = torch.load(os.path.join(WEIGHTS_DIR, fname), map_location='cpu', weights_only=False)
    if 'subject' in ckpt:
        checkpoints.append(ckpt)

if checkpoints:
    sids = [subject_id(c['subject']) for c in checkpoints]
    accs = [c['test_acc'] * 100 for c in checkpoints]
    mean = np.mean(accs)

    fig, ax = plt.subplots(figsize=(max(10, len(sids) * 0.45), 4))
    ax.bar(sids, accs, color='steelblue')
    ax.axhline(mean, color='tomato', linestyle='--', linewidth=1.5, label=f'Mean {mean:.1f}%')
    ax.set_ylim(0, 105)
    ax.set_xlabel('Test Subject')
    ax.set_ylabel('Test Accuracy (%)')
    ax.set_title(f'{MODEL_TYPE} LOSO — 8-Channel Efficient Feature Model  ({len(checkpoints)}/{len(subjects)} folds)')
    ax.legend()
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Compare 8-channel vs 24-channel feature-based baseline ────────────────────
baseline_weights_dir = os.path.abspath(
    f'../../cross_subject/feature_based/weights/{MODEL_TYPE}'
)

if os.path.isdir(baseline_weights_dir) and checkpoints:
    baseline_ckpts = []
    for fname in sorted(os.listdir(baseline_weights_dir)):
        if not fname.endswith('.pt'):
            continue
        ckpt = torch.load(os.path.join(baseline_weights_dir, fname),
                          map_location='cpu', weights_only=False)
        if 'subject' in ckpt:
            baseline_ckpts.append(ckpt)

    if baseline_ckpts:
        eff_map  = {subject_id(c['subject']): c['test_acc'] * 100 for c in checkpoints}
        base_map = {subject_id(c['subject']): c['test_acc'] * 100 for c in baseline_ckpts}
        common   = sorted(set(eff_map) & set(base_map))

        eff_accs  = [eff_map[s]  for s in common]
        base_accs = [base_map[s] for s in common]
        drops     = [b - e for b, e in zip(base_accs, eff_accs)]

        print(f'\n{MODEL_TYPE} — 24-channel vs 8-channel feature model  ({len(common)} subjects)')
        print(f'  24-channel mean : {np.mean(base_accs):.2f}%')
        print(f'   8-channel mean : {np.mean(eff_accs):.2f}%')
        print(f'  Accuracy drop   : {np.mean(drops):+.2f} pp  (mean per subject)')

        x = np.arange(len(common))
        w = 0.38
        fig, ax = plt.subplots(figsize=(max(12, len(common) * 0.5), 4))
        ax.bar(x - w/2, base_accs, w, label='24-channel baseline', color='steelblue')
        ax.bar(x + w/2, eff_accs,  w, label='8-channel efficient',  color='darkorange')
        ax.set_xticks(x)
        ax.set_xticklabels(common, rotation=45, ha='right', fontsize=8)
        ax.set_ylim(0, 105)
        ax.set_xlabel('Test Subject')
        ax.set_ylabel('Test Accuracy (%)')
        ax.set_title(f'{MODEL_TYPE}: 24-channel vs 8-channel Feature-Based LOSO')
        ax.legend()
        plt.tight_layout()
        plt.show()
else:
    print('Run training first, or ensure cross_subject/feature_based results exist for comparison.')